[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/rare_variant_SAIGE_reveal.ipynb)

# 🧬 Gene-Based Association Testing with SAIGE-Gene

SAIGE-Gene is an extension of the SAIGE software specifically designed for performing rare variant association tests. It is particularly powerful because it scales to massive cohorts (like the UK Biobank) while effectively controlling for sample relatedness, population stratification, and highly unbalanced case-control ratios.

In this practical, we will walk through the two core steps of a SAIGE-Gene analysis using a toy dataset, and then visualize pre-computed real-world results.

## 🛠️ Environment Initialization and SAIGE Install

To run SAIGE inside this notebook, we need to load the R computing environment and install the SAIGE software directly from its source code using a fast package manager called `pixi`.

Run the setup blocks below to prepare your environment. *(This may take a minute!)*

In [ ]:
# Load the R magic extension so we can write native R inside this notebook
%load_ext rpy2.ipython

In [ ]:
# Clone the source code repository for SAIGE-gene
!git clone https://github.com/saigegit/SAIGE.git

In [ ]:
# 1. Install the Pixi package manager
!curl -fsSL https://pixi.sh/install.sh | bash

# 2. Add Pixi to the system PATH so the notebook can find the command
import os
os.environ['PATH'] += ':/root/.pixi/bin'

# 3. Verify the installation
!pixi --version

In [ ]:
%cd /content/SAIGE
!pixi install

In [ ]:
import os
os.chmod('/content/SAIGE/extdata/step1_fitNULLGLMM.R', 0o755)
os.chmod('/content/SAIGE/extdata/step2_SPAtests.R', 0o755)
os.chmod('/content/SAIGE/extdata/createSparseGRM.R', 0o755)

> ⚠️ **CRITICAL CHECKPOINT:** The installation code blocks above must finish running without errors before proceeding. Wait for the green checkmarks to appear next to each cell!

---

# 🔵 Step 1: Fitting the Null Mixed Model

Before testing any specific genes, SAIGE requires us to fit a **Null Generalized Linear Mixed Model (GLMM)**. 

This step calculates the baseline variance of our phenotype *without* considering the genetic markers we want to test. It accounts for covariates (like Age, Sex, and Principal Components) and uses a **Genetic Relationship Matrix (GRM)** to adjust for any underlying relatedness or population structure among our samples.

First, we'll need to create a sparse GRM. This takes a randomly sampled set of relatively common markers that have been LD-pruned, and calculates the relatedness between all individuals. The 'sparse' matrix implies that unless individuals are more related than a certain cutoff (which we can specify), their relatedness is forced to 0. This gives a massive speedup when running on biobank scale data!

Run the following command to generate the sparse GRM:

In [ ]:
%cd /content/SAIGE
!pixi run Rscript extdata/createSparseGRM.R \
  --plinkFile=extdata/input/nfam_100_nindep_0_step1_includeGenoFile_100markers \
  --nThreads=1 \
  --outputPrefix=extdata/output/sparseGRM \
  --numRandomMarkerforSparseKin=2000 \
  --relatednessCutoff=0.05

### Reviewing the Command Options

Now that you've got a sparse GRM, we can fit a null model. Here are the options that you can pass to the step 1 R script.

In [ ]:
!pixi run Rscript extdata/step1_fitNULLGLMM.R --help

Let's run a script to perform step 1 of SAIGE, and fit the null model.

In [ ]:
!pixi run Rscript extdata/step1_fitNULLGLMM.R \
  --plinkFile=extdata/input/nfam_100_nindep_0_step1_includeGenoFile_100markers \
  --phenoFile=extdata/input/pheno_1000samples.txt_withdosages_withBothTraitTypes.txt \
  --phenoCol=y_binary \
  --covarColList=x1,x2 \
  --sampleIDColinphenoFile=IID \
  --traitType=binary \
  --outputPrefix=extdata/output/example_binary_sparseGRM \
  --nThreads=1 \
  --IsSparseKin=TRUE \
  --sparseGRMFile=extdata/output/sparseGRM_relatednessCutoff_0.05_2000_randomMarkersUsed.sparseGRM.mtx \
  --sparseGRMSampleIDFile=extdata/output/sparseGRM_relatednessCutoff_0.05_2000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt > step1_fitNULLGLMM.log

Take a look in `step1_fitNULLGLMM.log`:

In [ ]:
!cat step1_fitNULLGLMM.log

If everything has worked according to plan, the log should end with `* DONE (SAIGE)` and you should now have a saved null model ready for testing!

---

# 🟣 Step 2: Executing Gene-Based & Set-Based Association Tests

With our null model established, we can read a target marker file containing genotypes or dosages (supporting `VCF`, `BCF`, `BGEN`, or `SAV` - we'll use `BGEN` today) alongside a **group specification file** to run aggregated gene-level calculations like SKAT, Burden, and SKAT-O tests. Here's all the input files we need for step 2:

## Input files
####Dosage/genotype file containing dosages or genotypes of markers to test
`extdata/input/genotype_100markers.bgen`
`extdata/input/genotype_100markers.bgen.bgi`

The `.bgi` file is an index - it allows huge speedups if we want to zip to different parts of the bgen or slice and dice the data.

####Sample file

This file contains one column for sample IDs corresponding to the sample order in the dosage file. No header is included. The file is ONLY for `BGEN` input.

`extdata/input/samplelist.txt`

####Model file from step 1
`extdata/output/example_binary_sparseGRM.rda`

####Variance ratio file from step 1

`extdata/output/example_binary_sparseGRM.varianceRatio.txt`

#### Group File (specific to SAIGE-gene)

`extdata/input/group_new_chrposa1a2.txt`

### Understanding the Group File Format
A group mapping file structures variants into specific genomic sets (e.g., matching a gene identifier). It contains at least two lines for each gene/set. The first two lines are required:
1.  **`var` line:** The gene name followed by structural variant identifiers.
2.  **`anno` line:** The functional annotations matching each variant sequence item.
Optionally, you can include a third line representing the 'weights' for each marker:
3.  **`weight` line** *(optional)***:** Explicit weights per marker. If missing, SAIGE dynamically weights variants via a Beta distribution profile based on Minor Allele Frequency: $\text{Beta}(\text{MAF}, 1, 25)$.

Take a look in the group file that we're going to use for this practical:
Try writing the R code to read and print this file (`extdata/input/group_new_chrposa1a2.txt`) in the empty cell below!

<details>
  <summary>💡 Stuck? Click here to reveal the code solution!</summary>

  ```R
  %%R
  groupfile <- readLines("extdata/input/group_new_chrposa1a2.txt")
  print(groupfile)
  ```
</details>

Does the output make sense?

Can you go into R (or python if you're so inclined) and plot out the $\text{Beta}(x, 1, 25)$ distribution between $x=0$ and $x=1$? 

In [ ]:
%%R
x <- seq(0, 1, length.out=1000)
y <- dbeta(x, 1, 25)
plot(x, y, type="l", col="blue", lwd=2, xlab="MAF", ylab="Density", main="Beta(1, 25) Distribution")

Does this make sense? Why?

In [ ]:
# View parameter descriptions for the Step 2 association script
!pixi run Rscript extdata/step2_SPAtests.R --help

In [ ]:
!pixi run Rscript extdata/step2_SPAtests.R \
  --bgenFile=extdata/input/genotype_100markers.bgen \
  --bgenFileIndex=extdata/input/genotype_100markers.bgen.bgi \
  --sampleFile=extdata/input/samplelist.txt \
  --minMAF=0 \
  --minMAC=0.5 \
  --sampleFile=extdata/input/samplelist.txt \
  --GMMATmodelFile=extdata/output/example_binary_sparseGRM.rda \
  --varianceRatioFile=extdata/output/example_binary_sparseGRM.varianceRatio.txt \
  --SAIGEOutputFile=extdata/output/genotype_100markers_bgen_groupTest_out.txt \
  --groupFile=extdata/input/group_new_chrposa1a2.txt \
  --sparseGRMFile=extdata/output/sparseGRM_relatednessCutoff_0.05_2000_randomMarkersUsed.sparseGRM.mtx \
  --sparseGRMSampleIDFile=extdata/output/sparseGRM_relatednessCutoff_0.05_2000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt \
  --annotation_in_groupTest="synonymous,missense,pLoF" \
  --maxMAF_in_groupTest=0.0001,0.001,0.01 \
  --is_output_moreDetails=TRUE > step2_SPAtests.log

The screen output ends with the following text if it runs successfully:

```
Analysis finished.
[1] "* DONE (SAIGE)"
```

In [ ]:
!cat step2_SPAtests.log

Take a look in the output file for the gene-based tests.

What are each of the entries showing?
Try loading and printing this file (`extdata/output/genotype_100markers_bgen_groupTest_out.txt`) using `data.table` in the empty cell below!

<details>
  <summary>💡 Stuck? Click here to reveal the code solution!</summary>

  ```R
  %%R
  library(data.table)

  dt <- fread("extdata/output/genotype_100markers_bgen_groupTest_out.txt")
  print(dt)
  ```
</details>

---

# 📈 Visualizing Real UK Biobank Results

Congratulations! You've just run a full mixed-model rare variant association pipeline.

Because running this across the entire genome for 500,000 UK Biobank participants takes a massive compute cluster, we have pre-computed a real SAIGE-Gene output file for you to analyze. Let's pull down the dataset and create some publication-quality visualizations.

In [ ]:
!python -m pip -q install gdown
!gdown 12ZgP8uQpB9hB2X8T0zB0b95V0g4aA0aE -O /content/pilot-traits_uk-biobank_gene_uk-biobank.palmer.pilot.AFib.JULY23Freeze.ALL.EUR.28671.373704.SAIGE.gene.20240110.txt.gz

### ❓ Question 2.1
Take a look at the actual output file we're providing. Which trait is being tested?

<details>
  <summary>💡 Click here to reveal!</summary>
  
  If you look closely at the filename (`pilot.AFib`), the trait being tested is **Atrial Fibrillation (AFib)** in the European (EUR) cohort of the UK Biobank.
</details>

### ❓ Question 2.3
Take a quick look at the raw rows in the file using `zcat`. Which types of annotations have been evaluated?

<details>
  <summary>💡 Click here to reveal!</summary>
  
  SAIGE-gene evaluates sets based on the functional consequences of the variants. Looking at the `Group` column, you can see categories like:
  * `pLoF` (Predicted Loss of Function)
  * `damaging_missense_or_protein_altering`
  * `other_missense_or_protein_altering`
  * `synonymous` (Often used as our negative control!)
</details>

### 📊 Question 2.4: Generating a Quantile-Quantile (QQ) Plot
How do the results in synonymous variation look? Can you create a QQ plot?

Let's write a script to build a high-resolution QQ plot of our Synonymous variant groupings using `ggplot2` to inspect the distribution of observed $P$-values against expectation.
Try writing the `ggplot2` code to generate this plot in the empty cell below!

<details>
  <summary>💡 Stuck? Click here to reveal the code solution!</summary>

  ```R
%%R -w 12 -h 6 --units in -r 120
library(data.table)
library(dplyr)
library(tidyr)
library(ggplot2)

# 1. Load Data Safely
setwd("/content")
dt <- fread(cmd = "zcat pilot-traits_uk-biobank_gene_uk-biobank.palmer.pilot.AFib.JULY23Freeze.ALL.EUR.28671.373704.SAIGE.gene.20240110.txt.gz")

# 2. Advanced Data Pipeline with Ordered Factors and Confidence Intervals
plot_data <- dt %>%
  filter(Group != "Cauchy") %>%
  pivot_longer(
    cols = c(Pvalue, Pvalue_Burden, Pvalue_SKAT),
    names_to = "class",
    values_to = "Pvalue"
  ) %>%
  mutate(
    class = case_match(
      class,
      "Pvalue"        ~ "SKAT-O",
      "Pvalue_Burden" ~ "Burden",
      "Pvalue_SKAT"   ~ "SKAT"
    ),
    Group = case_match(
      Group,
      "damaging_missense_or_protein_altering" ~ "Damaging missense",
      "other_missense_or_protein_altering"    ~ "Other missense",
      "pLoF"                                  ~ "pLoF",
      "pLoF;damaging_missense_or_protein_altering" ~ "pLoF, damaging missense",
      "pLoF;damaging_missense_or_protein_altering;other_missense_or_protein_altering;synonymous" ~ "pLoF, damaging missense, other missense, synonymous",
      "synonymous"                            ~ "Synonymous"
    ),
    # Create the clean string label
    max_MAF_label = paste0("Maximum MAF Cutoff: ", max_MAF)
  ) %>%
  filter(Group == "Synonymous") %>%
  # --- FIX 1: Sort by numeric MAF descending to lock in your custom facet order ---
  arrange(desc(max_MAF)) %>%
  mutate(max_MAF_label = factor(max_MAF_label, levels = unique(max_MAF_label))) %>%
  # Process mathematical ranks and null distributions
  group_by(class, Group, max_MAF) %>%
  arrange(Pvalue, .by_group = TRUE) %>%
  mutate(
    rank = row_number(),
    n_total = n(),
    Pvalue_exp = rank / (n_total + 1),
    lower_ci = qbeta(0.025, rank, n_total - rank + 1),
    upper_ci = qbeta(0.975, rank, n_total - rank + 1)
  ) %>%
  ungroup()

p <- ggplot(plot_data, aes(x = -log10(Pvalue_exp), y = -log10(Pvalue))) +

  # Identity Line
  geom_abline(intercept = 0, slope = 1, color = "grey50", linetype = "dashed", linewidth = 0.7) +
  geom_ribbon(
    data = filter(plot_data, class == "SKAT-O"),
    aes(x = -log10(Pvalue_exp), ymin = -log10(upper_ci), ymax = -log10(lower_ci)),
    fill = "grey85",
    alpha = 0.5,
    inherit.aes = FALSE
  ) +

  geom_point(aes(color = class), size = 1.8, alpha = 0.65) +

  # Multi-panel grid layout
  facet_wrap(~max_MAF_label, scales = 'free') +
  scale_color_brewer(palette = "Set1") +

  # Fine-Tuning Presentation Theme
  theme_minimal(base_size = 13) +
  theme(
    panel.border       = element_rect(color = "grey80", fill = NA, linewidth = 0.8),
    panel.grid.minor   = element_blank(),
    strip.background   = element_rect(fill = "grey96", color = "grey80", linewidth = 0.5),
    strip.text         = element_text(face = "bold", color = "grey20", size = 11),
    legend.position    = "bottom",
    legend.title       = element_text(face = "bold"),
    legend.background  = element_rect(fill = "white", color = "grey90", linewidth = 0.3),
    plot.title         = element_text(face = "bold", size = 15, margin = margin(b = 6)),
    plot.subtitle      = element_text(color = "grey45", size = 10, margin = margin(b = 15))
  ) +
  labs(
    title    = "Quantile-Quantile (QQ) Plot with 95% Confidence Intervals",
    subtitle = "Synonymous Gene-level Testing Across MAF Cutoffs",
    x        = expression(Expected~-log[10](italic(p))),
    y        = expression(Observed~-log[10](italic(p))),
    color    = "Statistical Test"
  )
print(p)
  ```
</details>

Discuss these plots in your group. What do you notice?

<details>
  <summary>💡 Click to reveal the explanation!</summary>
  
  You should have noticed that there's lift off from $y=x$ for synonymous variation, suggesting that there could well be common variant associations nearby that you're tagging. It might be a good idea to look and see if there are common variant associations close to those lead gene-phenotype associations.
</details>

---
## 🎉 End of SAIGE-Gene Practical
Great job! You have successfully installed SAIGE, run a null mixed model to control for population structure, performed rare variant group testing, and visualized the summary statistics of a real biobank-scale GWAS.